In [ ]:
!pip install -q transformers sentencepiece accelerate torch

In [ ]:
import torch

from transformers import AutoTokenizer
from transformers import AutoModelForSeq2SeqLM

In [ ]:
MODEL_NAME = "facebook/nllb-200-distilled-600M"

In [ ]:
LANGUAGE_CODES = {
    "English": "eng_Latn",
    "French": "fra_Latn",
    "Spanish": "spa_Latn",
    "Hindi": "hin_Deva",
    "Tamil": "tam_Taml"
}

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    device_map="auto"
)

print("Model loaded successfully")

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 4.85MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.3MB            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.46GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Model loaded successfully


In [ ]:
import re

def preprocess_text(text):

    # Convert input to string
    text = str(text)

    # Remove spaces from beginning and end
    text = text.strip()

    # Replace multiple spaces with a single space
    text = re.sub(r"\s+", " ", text)

    return text

In [ ]:
def extract_error_codes(text):

    error_codes = re.findall(
        r'\b[A-Z]+_\d+\b',
        text
    )

    for error_code in error_codes:
        text = text.replace(error_code, "")

    # Clean extra spaces created after removing error code
    text = re.sub(r"\s+", " ", text).strip()

    return text, error_codes

In [ ]:
text = "My payment is not working!!! 😡 ERR_401"

clean_text = preprocess_text(text)

translation_text, error_codes = extract_error_codes(
    clean_text
)

print("Text sent to NLLB:")
print(translation_text)

print("\nProtected Error Codes:")
print(error_codes)

Text sent to NLLB:
My payment is not working!!! 😡

Protected Error Codes:
['ERR_401']


In [ ]:
def protect_error_codes(text):

    protected_terms = {}

    error_codes = re.findall(
        r'\b[A-Z]+_\d+\b',
        text
    )

    for index, error_code in enumerate(error_codes):

        placeholder = f"__ERROR_{index}__"

        text = text.replace(
            error_code,
            placeholder
        )

        protected_terms[placeholder] = error_code

    return text, protected_terms

In [ ]:
sample_text = "   My payment   is not working!!! 😡 ERR_401   "

clean_text = preprocess_text(sample_text)

print("Original :", sample_text)
print("Processed:", clean_text)

Original :    My payment   is not working!!! 😡 ERR_401   
Processed: My payment is not working!!! 😡 ERR_401


In [ ]:
protected_text, protected_terms = protect_error_codes(
    clean_text
)

print("Protected Text:")
print(protected_text)

print("\nProtected Terms:")
print(protected_terms)

Protected Text:
My payment is not working!!! 😡 __ERROR_0__

Protected Terms:
{'__ERROR_0__': 'ERR_401'}


In [ ]:
source_lang = "eng_Latn"

tokenizer.src_lang = source_lang

In [ ]:


tokenizer.src_lang = "eng_Latn"

inputs = tokenizer(
    translation_text,
    return_tensors="pt"
)

tokens = tokenizer.tokenize(translation_text)

print("Original text:")
print(sample_text)

print("\nClean text:")
print(translation_text)

print("\nTokens:")
print(tokens)

print("\nInput IDs:")
print(inputs["input_ids"])

print("\nAttention Mask:")
print(inputs["attention_mask"])

Original text:
   My payment   is not working!!! 😡 ERR_401   

Clean text:
My payment is not working!!! 😡

Tokens:
['▁My', '▁payment', '▁is', '▁not', '▁working', '!!!', '▁', '<unk>']

Input IDs:
tensor([[256047,   7177, 117325,    248,   2294,  46945,  10660, 248059,      3,
              2]])

Attention Mask:
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


In [ ]:
# Move input tensors to model device
inputs = {
    key: value.to(model.device)
    for key, value in inputs.items()
}

# Target language
target_lang = "tam_Taml"

# Convert target language token to ID
target_token_id = tokenizer.convert_tokens_to_ids(
    target_lang
)

# Generate translated token IDs
generated_tokens = model.generate(
    **inputs,
    forced_bos_token_id=target_token_id,
    max_length=256
)

In [ ]:
print(generated_tokens)

tensor([[     2, 256170,  11264, 141443,  48674,  20487,  24773, 248075,      2]],
       device='cuda:0')


In [ ]:
# Decode generated token IDs
decoded_output = tokenizer.batch_decode(
    generated_tokens,
    skip_special_tokens=True
)

# Get first translated sentence
translated_text = decoded_output[0]

print("Translated Text:")
print(translated_text)

Translated Text:
என் பணம் வேலை செய்யவில்லை!


In [ ]:
def restore_error_codes(translated_text, error_codes):

    if error_codes:
        translated_text = (
            translated_text
            + " "
            + " ".join(error_codes)
        )

    return translated_text

In [ ]:
final_text = restore_error_codes(
    translated_text,
    error_codes
)

print("Final Translation:")
print(final_text)

Final Translation:
என் பணம் வேலை செய்யவில்லை! ERR_401


In [ ]:
def translate_text(text, source_lang, target_lang, max_length=256):

    # Step 1 - Preprocess
    clean_text = preprocess_text(text)

    # Step 2 - Extract and protect error codes
    translation_text, error_codes = extract_error_codes(
        clean_text
    )

    # Step 3 - Set source language
    tokenizer.src_lang = source_lang

    # Step 4 - Tokenization
    inputs = tokenizer(
        translation_text,
        return_tensors="pt"
    )

    # Step 5 - Move input tensors to model device
    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    # Step 6 - Convert target language to token ID
    target_token_id = tokenizer.convert_tokens_to_ids(
        target_lang
    )

    # Step 7 - Generate translation
    generated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=target_token_id,
        max_length=max_length
    )

    # Step 8 - Decode token IDs to text
    translated_text = tokenizer.batch_decode(
        generated_tokens,
        skip_special_tokens=True
    )[0]

    # Step 9 - Restore error codes
    final_text = restore_error_codes(
        translated_text,
        error_codes
    )

    return final_text

In [ ]:
translate_text(
    "My payment is not working. ERR_401",
    "eng_Latn",
    "hin_Deva"
)

'मेरा भुगतान काम नहीं कर रहा है। ERR_401'

In [ ]:
test_cases = [
    {
        "text": "My payment is not working. ERR_401",
        "source_lang": "eng_Latn",
        "target_lang": "tam_Taml",
        "language": "Tamil"
    },

    {
        "text": "My payment is not working. ERR_401",
        "source_lang": "eng_Latn",
        "target_lang": "hin_Deva",
        "language": "Hindi"
    },

    {
        "text": "My payment is not working. ERR_401",
        "source_lang": "eng_Latn",
        "target_lang": "fra_Latn",
        "language": "French"
    },

    {
        "text": "My payment is not working. ERR_401",
        "source_lang": "eng_Latn",
        "target_lang": "spa_Latn",
        "language": "Spanish"
    }
]

In [ ]:
for case in test_cases:

    result = translate_text(
        case["text"],
        case["source_lang"],
        case["target_lang"]
    )

    print("Target Language:", case["language"])
    print("Original Text:", case["text"])
    print("Translated Text:", result)
    print("-" * 50)

Target Language: Tamil
Original Text: My payment is not working. ERR_401
Translated Text: என் பணம் வேலை செய்யவில்லை. ERR_401
--------------------------------------------------
Target Language: Hindi
Original Text: My payment is not working. ERR_401
Translated Text: मेरा भुगतान काम नहीं कर रहा है। ERR_401
--------------------------------------------------
Target Language: French
Original Text: My payment is not working. ERR_401
Translated Text: Mon paiement ne fonctionne pas. ERR_401
--------------------------------------------------
Target Language: Spanish
Original Text: My payment is not working. ERR_401
Translated Text: Mi pago no está funcionando. ERR_401
--------------------------------------------------


In [ ]:
!pip install -q langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 15.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
from langdetect import detect

In [ ]:
LANGUAGE_MAP = {
    "en": "eng_Latn",
    "ta": "tam_Taml",
    "hi": "hin_Deva",
    "fr": "fra_Latn",
    "es": "spa_Latn"
}

In [ ]:
def detect_source_language(text):

    detected_lang = detect(text)

    if detected_lang not in LANGUAGE_MAP:
        raise ValueError(
            f"Unsupported source language: {detected_lang}"
        )

    return LANGUAGE_MAP[detected_lang]

In [ ]:
detected_lang = detect(text)

In [ ]:
LANGUAGE_MAP[detected_lang]

'eng_Latn'

In [ ]:
def translate_text(text, target_lang, max_length=256):

    # Step 1 - Preprocess
    clean_text = preprocess_text(text)

    # Step 2 - Detect source language
    source_lang = detect_source_language(clean_text)

    # Step 3 - Extract error codes
    translation_text, error_codes = extract_error_codes(
        clean_text
    )

    # Step 4 - Set detected source language
    tokenizer.src_lang = source_lang

    # Step 5 - Tokenization
    inputs = tokenizer(
        translation_text,
        return_tensors="pt"
    )

    # Step 6 - Move tensors to model device
    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    # Step 7 - Target language token ID
    target_token_id = tokenizer.convert_tokens_to_ids(
        target_lang
    )

    # Step 8 - Translation
    generated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=target_token_id,
        max_length=max_length
    )

    # Step 9 - Decode
    translated_text = tokenizer.batch_decode(
        generated_tokens,
        skip_special_tokens=True
    )[0]

    # Step 10 - Restore error codes
    final_text = restore_error_codes(
        translated_text,
        error_codes
    )

    return final_text

In [ ]:
translate_text(
    text,
    "tam_Taml"
)

'என் பணம் வேலை செய்யவில்லை! ERR_401'

In [ ]:
robustness_tests = [
    {
        "type": "Normal",
        "text": "My payment is not working."
    },

    {
        "type": "Abbreviation / Informal",
        "text": "pls help payment nt wrking"
    },

    {
        "type": "Emoji",
        "text": "My payment failed 😭"
    },

    {
        "type": "Urgent Tone",
        "text": "URGENT!!! My account is blocked!"
    },

    {
        "type": "Technical Error Code",
        "text": "I am getting ERR_401 after login."
    },

    {
        "type": "Code-switched",
        "text": "Payment panna mudiyala please help"
    }
]

In [ ]:
for case in robustness_tests:

    try:
        result = translate_text(
            case["text"],
            "tam_Taml"
        )

        print("Test Type:", case["type"])
        print("Input:", case["text"])
        print("Output:", result)

    except Exception as e:

        print("Test Type:", case["type"])
        print("Input:", case["text"])
        print("Error:", e)

    print("-" * 60)

Test Type: Normal
Input: My payment is not working.
Output: என் பணம் வேலை செய்யவில்லை.
------------------------------------------------------------
Test Type: Abbreviation / Informal
Input: pls help payment nt wrking
Error: Unsupported source language: nl
------------------------------------------------------------
Test Type: Emoji
Input: My payment failed 😭
Error: Unsupported source language: cy
------------------------------------------------------------
Test Type: Urgent Tone
Input: URGENT!!! My account is blocked!
Output: அவசரமாக! என் கணக்கு தடுக்கப்பட்டுள்ளது!
------------------------------------------------------------
Test Type: Technical Error Code
Input: I am getting ERR_401 after login.
Error: Unsupported source language: da
------------------------------------------------------------
Test Type: Code-switched
Input: Payment panna mudiyala please help
Error: Unsupported source language: tl
------------------------------------------------------------


In [ ]:
for case in robustness_tests:

    try:
        cleaned = preprocess_text(case["text"])

        detected_source = detect_source_language(
            cleaned
        )

        result = translate_text(
            case["text"],
            "tam_Taml"
        )

        print("Test Type:", case["type"])
        print("Input:", case["text"])
        print("Detected Source:", detected_source)
        print("Output:", result)

    except Exception as e:

        print("Test Type:", case["type"])
        print("Input:", case["text"])
        print("Error:", e)

    print("-" * 60)

Test Type: Normal
Input: My payment is not working.
Detected Source: eng_Latn
Output: என் பணம் வேலை செய்யவில்லை.
------------------------------------------------------------
Test Type: Abbreviation / Informal
Input: pls help payment nt wrking
Detected Source: eng_Latn
Output: தயவுசெய்து பணம் செலுத்துவதற்கு உதவுங்கள்
------------------------------------------------------------
Test Type: Emoji
Input: My payment failed 😭
Detected Source: eng_Latn
Output: என் பணம் தோல்வியடைந்தது 😭
------------------------------------------------------------
Test Type: Urgent Tone
Input: URGENT!!! My account is blocked!
Detected Source: eng_Latn
Output: அவசரமாக! என் கணக்கு தடுக்கப்பட்டுள்ளது!
------------------------------------------------------------
Test Type: Technical Error Code
Input: I am getting ERR_401 after login.
Detected Source: eng_Latn
Output: நான் உள்நுழைவு பிறகு பெறுகிறேன். ERR_401
------------------------------------------------------------
Test Type: Code-switched
Input: Payment panna mud

In [ ]:
def detect_source_language(text):

    detected_lang = detect(text)

    if detected_lang in LANGUAGE_MAP:
        return LANGUAGE_MAP[detected_lang]

    # Fallback for noisy Latin-script support messages
    if re.search(r'[A-Za-z]', text):
        return "eng_Latn"

    raise ValueError(
        f"Unsupported source language: {detected_lang}"
    )

In [ ]:
!pip install -q sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 14.6 MB/s eta 0:00:00


In [ ]:
from sacrebleu.metrics import BLEU, CHRF

In [ ]:
evaluation_data = [
    {
        "source": "My payment was declined.",
        "reference": "Mon paiement a été refusé."
    },

    {
        "source": "Please reset my password.",
        "reference": "Veuillez réinitialiser mon mot de passe."
    },

    {
        "source": "My account is blocked.",
        "reference": "Mon compte est bloqué."
    },

    {
        "source": "I cannot login to my account.",
        "reference": "Je ne peux pas me connecter à mon compte."
    }
]

In [ ]:
predictions = []
references = []

In [ ]:
for item in evaluation_data:

    prediction = translate_text(
        item["source"],
        "fra_Latn"
    )

    predictions.append(prediction)
    references.append(item["reference"])

In [ ]:
from sacrebleu.metrics import BLEU, CHRF


evaluation_data = [
    {
        "source": "My payment was declined.",
        "reference": "Mon paiement a été refusé."
    },

    {
        "source": "Please reset my password.",
        "reference": "Veuillez réinitialiser mon mot de passe."
    },

    {
        "source": "My account is blocked.",
        "reference": "Mon compte est bloqué."
    },

    {
        "source": "I cannot login to my account.",
        "reference": "Je ne peux pas me connecter à mon compte."
    }
]


predictions = []
references = []


for item in evaluation_data:

    prediction = translate_text(
        item["source"],
        "fra_Latn"
    )

    predictions.append(prediction)
    references.append(item["reference"])

    print("Source:")
    print(item["source"])

    print("Prediction:")
    print(prediction)

    print("Reference:")
    print(item["reference"])

    print("-" * 50)


bleu = BLEU()

bleu_score = bleu.corpus_score(
    predictions,
    [references]
)


chrf = CHRF()

chrf_score = chrf.corpus_score(
    predictions,
    [references]
)


print("BLEU Score:")
print(bleu_score)

print("\nchrF Score:")
print(chrf_score)

Source:
My payment was declined.
Prediction:
Mon paiement a été refusé.
Reference:
Mon paiement a été refusé.
--------------------------------------------------
Source:
Please reset my password.
Prediction:
S'il vous plaît réinitialisez mon mot de passe.
Reference:
Veuillez réinitialiser mon mot de passe.
--------------------------------------------------
Source:
My account is blocked.
Prediction:
Mon compte est bloqué.
Reference:
Mon compte est bloqué.
--------------------------------------------------
Source:
I cannot login to my account.
Prediction:
Je ne peux pas me connecter à mon compte.
Reference:
Je ne peux pas me connecter à mon compte.
--------------------------------------------------
BLEU Score:
BLEU = 82.65 86.7/84.6/81.8/77.8 (BP = 1.000 ratio = 1.071 hyp_len = 30 ref_len = 28)

chrF Score:
chrF2 = 88.40


In [ ]:
def translate_text(text, target_lang, max_length=256):

    # Step 1 - Preprocess input
    clean_text = preprocess_text(text)

    # Step 2 - Detect source language
    source_lang = detect_source_language(clean_text)

    # Step 3 - Extract technical terms
    translation_text, protected_terms = extract_technical_terms(
        clean_text
    )

    # Step 4 - Set source language for tokenizer
    tokenizer.src_lang = source_lang

    # Step 5 - Tokenize
    inputs = tokenizer(
        translation_text,
        return_tensors="pt"
    )

    # Step 6 - Move tensors to same device as model
    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    # Step 7 - Convert target language code to token ID
    target_token_id = tokenizer.convert_tokens_to_ids(
        target_lang
    )

    # Step 8 - Generate translation
    generated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=target_token_id,
        max_length=max_length
    )

    # Step 9 - Decode generated token IDs
    translated_text = tokenizer.batch_decode(
        generated_tokens,
        skip_special_tokens=True
    )[0]

    # Step 10 - Restore protected technical terms
    final_text = restore_technical_terms(
        translated_text,
        protected_terms
    )

    return {
        "source_language": source_lang,
        "target_language": target_lang,
        "original_text": text,
        "translated_text": final_text
    }